# H1b Stage 1 specification audit (exploratory)

**Exploratory (ADR 0008 §9).** Not a test of the frozen H1b claim; 5 % is a descriptive
reference that does not inherit H1b's unused α (ADR 0008 §7.1). No model-implied benchmark is
computed here.

`notebooks/03_h1b_exploratory.ipynb` (commit 8fba5b6) is the **original Stage 1 run** and is
left unchanged. This notebook (1) re-derives its bounds results and asserts they match, (2)
records its exact specification, and (3) runs a **draft-design sensitivity analysis**: the
ARDL design proposed in the superseded ADR 0008 drafts (r2 §6.1–6.2, with the r3 §6.3
tax-lag correction), plus spec-specific mappings added during this audit. Each variant gets a
simulation of exactly the same selection and deterministic terms. The sensitivity variant is
**not** a stricter reading of frozen ADR 0008, and neither variant is preferred on the basis of
significance.

In [1]:
import functools

import numpy as np
import pandas as pd

from kagura import longrun as lr
from kagura.explore import ERAS

pd.set_option("display.width", 170, "display.max_columns", 40, "display.precision", 3)
df = lr.load_monthly()  # verifies all frozen checksums
pre = df.loc[lr.PRE_START : lr.PRE_END]
SPEC_VARS = {
    "(a) s on x, d2": ("s", ["x", "d2"], "1981-01"),
    "(b) q on d2": ("q", ["d2"], "1981-01"),
    "(c) s on x, d10": ("s", ["x", "d10"], "1986-07"),
    "(d) s on x_tax, d2": ("s", ["x_tax", "d2"], "1981-01"),
}
WORKERS, SIM_REPS = 10, 2000


def data(name):
    dep, regs, start = SPEC_VARS[name]
    sub = pre.loc[start:]
    return sub, sub[dep].to_numpy(float), sub[regs].to_numpy(float), regs

## 1. Original Stage 1 specification (as run in notebook 03), re-derived

| Element | Original implementation |
|---|---|
| Model | UECM, PSS Case III: constant, y_{t−1}, X_{t−1}, Δy lags 1..p−1, ΔX lags 0..q−1 |
| Deterministics | constant only: **no seasonal dummies, no tax impulses** |
| Lag grid | p ∈ 1..12 and **one common q** ∈ 1..12 for all regressors (144 models) |
| Sample | first row = 13th month: 1982-02…2021-12, **n = 479** ((c): 1987-08…, n = 413) |
| Selection | lowest AIC among models with BG(12) p > 0.05; if none, AIC minimum flagged |
| Simulated quantiles | the same `select_ardl` call (same grid, start row, constant only) on independent **driftless** Gaussian random walks of the same length; R = 2,000 |

In [2]:
ORIG = {
    "(a) s on x, d2": (4.080, -1.868),
    "(b) q on d2": (1.877, -0.980),
    "(c) s on x, d10": (3.513, -2.978),
    "(d) s on x_tax, d2": (3.974, -1.876),
}
orig = {}
for name in SPEC_VARS:
    sub, y, X, regs = data(name)
    f = lr.select_ardl(y, X)
    assert (round(f.f_stat, 3), round(f.t, 3)) == ORIG[name], (name, f.f_stat, f.t)
    orig[name] = f
print("Original bounds statistics reproduced exactly for all four specifications.")

Original bounds statistics reproduced exactly for all four specifications.


In [3]:
# Complete primary specification (a), original implementation
f = orig["(a) s on x, d2"]
sub, y, X, regs = data("(a) s on x, d2")
names = ["const", "s(-1)", "x(-1)", "d2(-1)"] + [f"Δs(-{i})" for i in range(1, f.p)]
names += [f"Δ{v}(-{i})" for v in regs for i in range(f.q)]
yv, z = lr._uecm_design(y, X, f.p, f.q, f.start)
b, e, zinv = lr._ols(yv, z)
s2 = e @ e / (len(e) - z.shape[1])
nw = int(np.floor(4 * (len(e) / 100) ** (2 / 9)))
se_ols, se_hac = np.sqrt(s2 * np.diag(zinv)), np.sqrt(np.diag(lr._hac_cov(z, e, zinv, nw)))
coef = pd.DataFrame(
    {"coef": b, "se OLS": se_ols, "se HAC": se_hac, "t OLS": b / se_ols}, index=names
)
print(
    f"(a) original: sample {sub.index[f.start]}…{sub.index[-1]}, n = {f.nobs}, "
    f"p = {f.p}, q = {f.q}, "
    f"R² = {1 - e @ e / ((yv - yv.mean()) @ (yv - yv.mean())):.3f}, σ = {np.sqrt(s2):.3f}"
)
print(coef.to_string())
print(f"\nF = {f.f_stat:.3f}, t = {f.t:.3f}, φ = {f.phi:.4f}; BG(12) p = {f.bg_p:.3f}")
print(
    "Long run (−π/φ): "
    + ", ".join(
        f"{v} = {c:.3f} (HAC {h:.3f}, OLS {o:.3f})"
        for v, c, h, o in zip(regs, f.long_run, f.lr_se_hac, f.lr_se_ols, strict=True)
    )
)

(a) original: sample 1982-02…2021-12, n = 479, p = 12, q = 1, R² = 0.080, σ = 3.021
          coef  se OLS  se HAC  t OLS
const    5.904   4.122   4.439  1.432
s(-1)   -0.014   0.008   0.008 -1.868
x(-1)   -0.004   0.010   0.008 -0.455
d2(-1)   0.230   0.090   0.086  2.549
Δs(-1)   0.017   0.046   0.046  0.374
Δs(-2)   0.071   0.046   0.056  1.544
Δs(-3)   0.020   0.046   0.041  0.431
Δs(-4)   0.015   0.046   0.043  0.332
Δs(-5)  -0.053   0.045   0.057 -1.157
Δs(-6)  -0.078   0.045   0.038 -1.734
Δs(-7)   0.011   0.045   0.045  0.235
Δs(-8)   0.043   0.045   0.042  0.966
Δs(-9)   0.072   0.045   0.044  1.595
Δs(-10)  0.045   0.045   0.049  1.009
Δs(-11)  0.105   0.045   0.047  2.315
Δx(-0)  -0.419   0.328   0.308 -1.281
Δd2(-0)  0.849   0.444   0.617  1.913

F = 4.080, t = -1.868, φ = -0.0143; BG(12) p = 0.260
Long run (−π/φ): x = -0.301 (HAC 0.726, OLS 0.784), d2 = 15.999 (HAC 11.135, OLS 10.985)


## 2. Draft-design sensitivity variants

Where each element comes from. Frozen ADR 0008 §9.4 requires only "ARDL/UECM: Case III,
BG-filtered AIC lags, maximum 12"; both the original run and the sensitivity variant satisfy it.

| Element | Draft-design sensitivity variant | Source |
|---|---|---|
| Case III UECM, BG(12)-filtered AIC, lags ≤ 12 | as in the original | **Frozen ADR 0008 §9.4** |
| Deterministics | constant + **11 centred calendar-month dummies** + **tax impulses** I_{e+j} for e ∈ {1989-04, 1997-04, 2014-04, 2019-10}, j = 0..q_x − 1 (varying with the candidate q_x) | Superseded drafts: r2 §6.1 (dummies), r3 §6.3 option (i) (lagged impulses) |
| Lag grid | **independent** (p, q_x, q_d) ∈ {1..12}³ (1,728 models); (b): (p, q_d) ∈ {1..12}² | Superseded draft r2 §6.2 |
| Sample | first row = 13th month index 12: 1982-01…2021-12, **n = 480** ((c): 1987-07…, n = 414) | Superseded draft r2 §6.1 |
| Simulation with the same deterministic terms | same `select_ardl` call | Superseded draft r3 §2.2 |
| No BG-passing candidate | AIC minimum flagged rather than stopping (never triggered) | Added in this audit (exploratory) |

Spec-specific mappings, **added in this audit** (the drafts define the design for (a) only):
- (c): same design as (a) on the 10Y sample.
- (d): x_tax already removes the tax steps, so **seasonal dummies only, no tax impulses**.
- (b): the tax spike sits in the dependent variable Δq and its p − 1 lags, so impulses use
  j = 0..p − 1. The permanent tax step in q's level is not modelled (≈1–2 log-points against
  a q standard deviation of ≈ 20).

Simulated quantiles use exactly the same call (`select_ardl` with the same `start`,
`independent=True` and `terms`) on driftless random walks with the same calendar index.

In [4]:
TAX = {
    "(a) s on x, d2": "q0",
    "(b) q on d2": "p",
    "(c) s on x, d10": "q0",
    "(d) s on x_tax, d2": "none",
}
prot, sim_prot, sim_orig = {}, {}, {}
for name in SPEC_VARS:
    sub, y, X, regs = data(name)
    terms = functools.partial(lr.calendar_terms, sub.index, tax_lags=TAX[name])
    prot[name] = lr.select_ardl(y, X, start=12, independent=True, terms=terms)
    sim_prot[name] = lr.simulate_bounds_quantiles(
        len(y),
        len(regs),
        SIM_REPS,
        seed=20261025,
        start=12,
        independent=True,
        terms=terms,
        workers=WORKERS,
    )
    key = (len(y), len(regs))
    if key not in sim_orig:  # the original notebook's exact call and seed
        sim_orig[key] = lr.simulate_bounds_quantiles(
            len(y), len(regs), SIM_REPS, seed=20261024 + len(y) + len(regs)
        )
print(
    "design-matched simulated 95 % quantiles (F, t):",
    {k: tuple(round(v, 3) for v in q) for k, q in sim_prot.items()},
)
print(
    "original-procedure simulated 95 % quantiles (F, t):",
    {k: tuple(round(v, 3) for v in q) for k, q in sim_orig.items()},
)

design-matched simulated 95 % quantiles (F, t): {'(a) s on x, d2': (4.786, -3.538), '(b) q on d2': (5.628, -3.216), '(c) s on x, d10': (4.741, -3.487), '(d) s on x_tax, d2': (4.9, -3.574)}
original-procedure simulated 95 % quantiles (F, t): {(492, 2): (4.934, -3.539), (492, 1): (5.819, -3.251), (426, 2): (4.993, -3.63)}


## 3. Reconciliation table: original vs draft-design sensitivity

PSS values are **asymptotic** (Tables CI(iii)/CII(iii), 5 %). Simulated values are
**finite-sample** quantiles of the full selection procedure under driftless random walks
(the empirical x drifts; that mismatch is common to both variants).

In [5]:
def row(name, f, simq):
    sub, _y, _x, regs = data(name)
    f_i0, f_i1, ti0, ti1 = lr.PSS_5PCT[len(regs)]
    f_sim, ts = simq
    if f.f_stat > max(f_i1, f_sim) and f.t < min(ti1, ts):
        verdict = "levels relation indicated (F and t)"
    elif f.f_stat < f_i0 or f.t > ti0:
        verdict = "no levels relation indicated (F or t inside I(0) bound)"
    else:
        verdict = "inconclusive"
    hl = lr.half_life(lr.ar_polynomial(f))
    out = {
        "sample": f"{sub.index[f.start]}…{sub.index[-1]}",
        "n": f.nobs,
        "p": f.p,
        "q": f.q,
        "F": f.f_stat,
        "F PSS 5% I0/I1 (asy)": f"{f_i0}/{f_i1}",
        "F sim95 (fs)": f_sim,
        "t": f.t,
        "t PSS 5% I0/I1 (asy)": f"{ti0}/{ti1}",
        "t sim5 (fs)": ts,
        "BG12 p": f.bg_p,
        "BG passed": f.bg_passed,
        "phi": f.phi,
        "half-life point (m)": hl,
        "reading": verdict,
    }
    for j, v in enumerate(regs):
        out[f"LR {v}"] = f"{f.long_run[j]:.3f} ({f.lr_se_hac[j]:.3f})"
    return out


tab = {}
for name in SPEC_VARS:
    sub, y, X, regs = data(name)
    tab[(name, "original")] = row(name, orig[name], sim_orig[(len(sub), len(regs))])
    tab[(name, "draft-design")] = row(name, prot[name], sim_prot[name])
recon = pd.DataFrame(tab)
print(recon.to_string())

                                                               (a) s on x, d2                                                                                                       (b) q on d2                                                                                                   (c) s on x, d10                                                                                                (d) s on x_tax, d2                                                         
                                                                     original                                             draft-design                                                 original                                             draft-design                                                 original                                             draft-design                                                 original                                             draft-design
sample                                        

In [6]:
# Complete primary specification (a), draft-design sensitivity variant (dummies not printed)
name = "(a) s on x, d2"
f = prot[name]
sub, y, X, regs = data(name)
terms = lr.calendar_terms(sub.index, f.p, f.q, "q0")
yv, z = lr._uecm_design(y, X, f.p, f.q, 12)
ex = terms[12:]
ex = ex[:, np.abs(ex).sum(0) > 0]
zz = np.column_stack([z, ex])
b, e, zinv = lr._ols(yv, zz)
s2 = e @ e / (len(e) - zz.shape[1])
nw = int(np.floor(4 * (len(e) / 100) ** (2 / 9)))
se_ols, se_hac = np.sqrt(s2 * np.diag(zinv)), np.sqrt(np.diag(lr._hac_cov(zz, e, zinv, nw)))
names = ["const", "s(-1)", "x(-1)", "d2(-1)"] + [f"Δs(-{i})" for i in range(1, f.p)]
names += [f"Δx(-{i})" for i in range(f.q[0])] + [f"Δd2(-{i})" for i in range(f.q[1])]
k = len(names)
coef = pd.DataFrame(
    {"coef": b[:k], "se OLS": se_ols[:k], "se HAC": se_hac[:k], "t OLS": b[:k] / se_ols[:k]},
    index=names,
)
print(
    f"(a) draft-design sensitivity: n = {f.nobs}, p = {f.p}, (q_x, q_d) = {f.q}, "
    f"{ex.shape[1]} dummy columns "
    f"(11 seasonal + {ex.shape[1] - 11} tax impulses), σ = {np.sqrt(s2):.3f}"
)
print(coef.to_string())
print(f"\nF = {f.f_stat:.3f}, t = {f.t:.3f}, φ = {f.phi:.4f}, BG(12) p = {f.bg_p:.3f}")
print(
    "Long run: "
    + ", ".join(
        f"{v} = {c:.3f} (HAC {h:.3f})"
        for v, c, h in zip(regs, f.long_run, f.lr_se_hac, strict=True)
    )
)

(a) draft-design sensitivity: n = 480, p = 3, (q_x, q_d) = (2, 2), 19 dummy columns (11 seasonal + 8 tax impulses), σ = 3.043
          coef  se OLS  se HAC  t OLS
const    3.071   4.001   4.257  0.768
s(-1)   -0.009   0.008   0.008 -1.201
x(-1)   -0.009   0.009   0.008 -1.044
d2(-1)   0.236   0.091   0.090  2.605
Δs(-1)   0.034   0.047   0.049  0.720
Δs(-2)   0.076   0.046   0.055  1.657
Δx(-0)  -0.374   0.419   0.397 -0.892
Δx(-1)   0.130   0.421   0.440  0.308
Δd2(-0)  0.957   0.462   0.626  2.071
Δd2(-1)  0.784   0.448   0.457  1.750

F = 3.353, t = -1.201, φ = -0.0090, BG(12) p = 0.051
Long run: x = -1.045 (HAC 1.662), d2 = 26.093 (HAC 24.824)


## 4. Layer D (stability) and estimator agreement for the draft-design sensitivity variants

ARDL at each variant's selected orders and terms, re-estimated in the ADR 0004 pre-period eras.
FM-OLS and D-OLS have no seasonal or tax terms in any draft, so their Stage 1 estimates
(notebook 03) are unchanged and are repeated here for comparison.

In [7]:
PRE_ERAS = {k: v for k, v in ERAS.items() if v[1] <= "2021-12-31"}
era_rows, cmp_rows = {}, {}
for name in SPEC_VARS:
    sub, y, X, regs = data(name)
    f = prot[name]
    terms_full = lr.calendar_terms(
        sub.index, f.p, f.q if isinstance(f.q, tuple) else (f.q,), TAX[name]
    )
    for era, (lo, hi) in PRE_ERAS.items():
        m = (sub.index >= pd.Period(lo[:7], "M")) & (sub.index <= pd.Period(hi[:7], "M"))
        if m.sum() < 60:
            continue
        g = lr.fit_ardl(y[m], X[m], f.p, f.q, 12, terms_full[m])
        era_rows[(name, era)] = {
            "n": g.nobs,
            "phi": g.phi,
            "t": g.t,
            **{
                f"LR {v}": f"{g.long_run[j]:.2f} ({g.lr_se_hac[j]:.2f})" for j, v in enumerate(regs)
            },
        }
    fm, do = lr.fmols(y, X), lr.dols(y, X)
    for j, v in enumerate(regs):
        cmp_rows[(name, v)] = {
            "ARDL original": orig[name].long_run[j],
            "ARDL draft-design": f.long_run[j],
            "FM-OLS": fm.coef[1 + j],
            "D-OLS": do.coef[1 + j],
            "FM se": fm.se[1 + j],
            "D-OLS se": do.se[1 + j],
        }
print(pd.DataFrame(era_rows).T.to_string())
print()
print(pd.DataFrame(cmp_rows).T.round(3).to_string())

                                n    phi      t               LR x             LR d2             LR d10        LR x_tax
(a) s on x, d2     1981–1989   96 -0.038 -1.076        3.77 (2.03)     22.61 (23.09)                NaN             NaN
                   1990–1998   96  0.008  0.154     -18.48 (87.02)   -79.05 (398.12)                NaN             NaN
                   1999–2012  156 -0.052 -2.305        0.74 (0.58)       7.74 (3.17)                NaN             NaN
                   2013–2021   96 -0.127 -2.674       -0.26 (0.56)      -1.07 (1.62)                NaN             NaN
(b) q on d2        1981–1989   96 -0.019 -1.183                NaN     47.19 (32.03)                NaN             NaN
                   1990–1998   96 -0.001 -0.049                NaN  241.28 (5794.02)                NaN             NaN
                   1999–2012  156 -0.057 -2.562                NaN       6.64 (2.21)                NaN             NaN
                   2013–2021   96 -0.077

## 5. Half-life

The block-bootstrap intervals in notebook 03 rebuild data from the fitted, stable ARDL, so
they cannot represent a unit root and are too narrow near one. **They are not used as
evidence.** Only point half-lives are shown (table in §3); with φ between about −0.01 and
−0.04 and no bounds evidence of error correction, persistence is weakly identified and the
point values are descriptive.

## 6. Classification (written judgment after reading the output above; not a test)

| Spec | Original (nb 03) | Draft-design sensitivity | Class (both) | Reasons |
|---|---|---|---|---|
| (a) s on x, d2 | F 4.08, t −1.87 | F 3.35, t −1.20 | **Not defensible** | t inside the I(0) bound in both; F moves from the inconclusive band to below the I(0) bound once seasonal/tax terms and independent lags are added; θ < 0 in both (−0.30, −1.04); γ 16 → 26 with HAC se 11 → 25; sensitivity-variant BG p = 0.051 (selection sits at the BG edge). |
| (b) q on d2 | F 1.88, t −0.98 | F 2.05, t −0.66 | **Not defensible** | Both statistics inside the I(0) bounds in both variants; γ HAC se 21 → 51. |
| (c) s on x, d10 | F 3.51, t −2.98 | F 2.89, t −2.76 | **Not defensible** | F below I(0) in both; the t-ratio that was inconclusive (−2.98) moves inside the I(0) bound (−2.76); θ = 0.12–0.15 with intervals excluding 1. |
| (d) s on x_tax, d2 | F 3.97, t −1.88 | F 3.17, t −1.29 | **Not defensible** | As (a): F falls below I(0), t inside I(0); θ = −0.27 / −0.80. |

**What changes under the sensitivity design:** F falls for (a), (c) and (d) but **rises for (b)**
(1.88 → 2.05, still below the I(0) bound); every |t| falls. Nothing crosses an I(1) bound or a
simulated quantile, and the design-matched simulated quantiles (F 4.74–4.90, t −3.49 to −3.57 for k = 2)
are close to the original ones (4.93–4.99, −3.54 to −3.63). The specification differences
therefore do not change any classification. Era estimates under the sensitivity variants show the
same pattern as before: adjustment strongest in 2013–21 (φ −0.08 to −0.16) with γ between −3
and +1, and explosive or zero adjustment in 1990–98 (φ −0.001 to +0.008, γ −276 to +241).

**Model-implied benchmark:** not justified under either variant (ADR 0008 §9.4).
